# EDA — Occurrences GBIF (Alsace)

Analyse exploratoire des occurrences téléchargées par `champiquest.data.gbif`.

Objectifs (cf. `reports/partie-1-prediction-pousse.md`, section 8) :
1. distribution temporelle (années, jour-de-l'année)
2. distribution spatiale (carte, clustering → biais d'échantillonnage)
3. profil tabulaire automatique (`fg-data-profiling`)


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Réutilise les chemins et la config du package (pas de duplication)
from champiquest.data.gbif import RAW_DIR, load_config

config = load_config()
# Métadonnées des espèces (nom, saison, habitat) pour les analyses suivantes
species_meta = {s["key"]: s for s in config["species"]}

frames = {f.stem: pd.read_csv(f) for f in sorted(RAW_DIR.glob("*.csv"))}
all_df = pd.concat(frames.values(), ignore_index=True)
all_df["date"] = pd.to_datetime(
    {"year": all_df["year"], "month": all_df["month"], "day": all_df["day"]},
    errors="coerce",
)
all_df["day_of_year"] = all_df["date"].dt.dayofyear

print("Espèces chargées :", len(frames))
print("Total lignes      :", len(all_df))
print(all_df.groupby("species")["year"].agg(["count", "min", "max"]))


In [ ]:
# Répartition par année : où se concentrent réellement les données ?
g = sns.displot(
    data=all_df,
    x="year",
    col="species",
    col_wrap=2,
    bins=25,
    height=3,
    aspect=1.4,
)
g.set_titles("{col_name}")
g.fig.suptitle("Observations par année et par espèce", y=1.03)
plt.show()


In [ ]:
# Jour-de-l'année : confirme les fenêtres de saison (morille = printemps, autres = automne)
g = sns.displot(
    data=all_df,
    x="day_of_year",
    col="species",
    col_wrap=2,
    bins=26,
    height=3,
    aspect=1.4,
)
g.set_titles("{col_name}")
g.fig.suptitle("Observations par jour de l'année", y=1.03)
plt.show()


In [ ]:
import folium

# Carte des observations : le clustering révèle le biais d'échantillonnage.
center = [all_df["decimalLatitude"].mean(), all_df["decimalLongitude"].mean()]
m = folium.Map(location=center, zoom_start=9)

colors = {
    "Boletus edulis": "red",
    "Imleria badia": "orange",
    "Cantharellus cibarius": "green",
    "Macrolepiota procera": "purple",
    "Morchella esculenta": "blue",
}

for name, grp in all_df.groupby("species"):
    fg = folium.FeatureGroup(name=name, show=True)
    for _, row in grp.iterrows():
        folium.CircleMarker(
            location=[row["decimalLatitude"], row["decimalLongitude"]],
            radius=2,
            color=colors.get(name, "gray"),
            fill=True,
            fill_opacity=0.5,
        ).add_to(fg)
    fg.add_to(m)

folium.LayerControl().add_to(m)
m


In [ ]:
from data_profiling import ProfileReport

# Profil tabulaire automatique (peut prendre quelques dizaines de secondes).
profile = ProfileReport(all_df, title="Occurrences GBIF — Alsace", minimal=True)
out = Path("../data/processed/gbif_profile.html")
out.parent.mkdir(parents=True, exist_ok=True)
profile.to_file(str(out))
print(f"Rapport écrit dans {out}")


## 💡 Idées de features (à implémenter dans `features/weather.py`)

### Météo — fenêtre courte (J-14, J-30)
- cumul précipitations (J-7, J-14, J-30)
- température moyenne / min / max (J-14)
- **choc thermique** : temp moyenne (J-21→J-10) − temp moyenne (J-10→J-0)
- **amplitude thermique** : moyenne de (Tmax − Tmin)

### Météo — fenêtre longue (J-60, été)
- cumul précipitations (J-60, J-90)
- **bilan hydrique cumulé** : cumul(precip − ET0) sur 30 / 60 / 90 j (proxy sécheresse)
- **classe « été sec/pluvieux »** : catégorique à partir du cumul juin-août (vs médiane)

### Environnement (statique)
- **altitude** (DEM, ~76 m/px)
- **pente + orientation** (DEM — versant nord = frais/humide)
- **type de forêt** (CLC — feuillus/résineux/mixte, à venir)
- **sol** (SoilGrids) : pH, sable/silt/argile, carbone organique

### Contexte
- **jour de l'année** — ⚠️ risque de fuite calendrier : plutôt pour la règle métier que comme feature du modèle
- **espèce** — seulement si on fait un modèle global un jour
